# BeamGuardAI — Kaggle Notebook

A high-beam headlight violation detector: YOLO26 (vehicle + headlight) → OpenCV brightness/glare analysis → tracking-based persistence scoring → license-plate OCR → SQLite evidence log.

This notebook is adapted from a general roadmap specifically for **Kaggle's free GPU environment**. The main differences from a local setup:

- No display server — `cv2.imshow()` will crash on Kaggle. Frames are shown with `matplotlib` instead, and evidence images are just written to disk with `cv2.imwrite()` (which needs no GUI).
- `/kaggle/input/` is **read-only** — your dataset has to be uploaded as a Kaggle Dataset and attached via "Add Input", not written to directly.
- `/kaggle/working/` is your only writable space (20 GB) and is wiped unless you **Save Version (commit)** before your session ends.
- Free GPU quota is about **30 hours/week**, sessions cap at **~9–12 hours**, and you get a P100 (16 GB) or 2×T4 (32 GB combined) depending on what's selected in Settings → Accelerator.
- Streamlit can't run as a live server inside a Kaggle notebook (no open ports), so the dashboard step is replaced with an in-notebook report, plus instructions to run the real dashboard locally once you've downloaded your results.

Work through the sections top to bottom over the weeks — you don't need to run everything in one sitting.

## 0. Check your GPU and session

In [ ]:
import torch, subprocess

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("No GPU attached (nvidia-smi not found). CPU-only steps will still work.")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


If this shows no GPU, go to **Settings (right panel) → Accelerator → GPU T4 x2 or P100**, then restart the session. Remember: the GPU quota only burns while a session is *running* with an accelerator attached — turn it off (Accelerator → None) whenever you're just writing code, not training.

## 1. Install packages

Kaggle pre-installs PyTorch and most data-science libraries, so this is a short list. `opencv-python-headless` avoids the GUI dependencies that fail on Kaggle.

In [ ]:
!pip install -q ultralytics opencv-python-headless easyocr


## 2. Project folders

Everything writable lives under `/kaggle/working`.

In [ ]:
import os

BASE = "/kaggle/working/BeamGuardAI"

folders = [
    "dataset/images/train", "dataset/images/val", "dataset/images/test",
    "dataset/labels/train", "dataset/labels/val", "dataset/labels/test",
    "models", "videos", "output/evidence", "output/processed_videos",
]

for f in folders:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

print("Created project structure under", BASE)


## 2b. Point the notebook at your Night Images set

Your dataset (400 night-road photos, no video, no labels) was uploaded as **Night Images set**, which Kaggle slugs as `night-images-set`. It contains a `nighttime_driving_dataset/` folder of `.jpg` files. This cell finds the folder automatically, so it still works if Kaggle named the slug slightly differently.

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt

DATASET_SLUG = "night-images-set"

def locate_images(slug=DATASET_SLUG):
    for root in (f"/kaggle/input/{slug}", "/kaggle/input"):
        paths = []
        for ext in ("jpg", "jpeg", "png"):
            paths += glob.glob(f"{root}/**/*.{ext}", recursive=True)
        if paths:
            paths = sorted(paths)
            return os.path.dirname(paths[0]), paths
    return None, []

DATA_DIR, IMG_PATHS = locate_images()
print("Image folder:", DATA_DIR)
print("Images found:", len(IMG_PATHS))
if not IMG_PATHS:
    print("Nothing found. Add the dataset via Add Input, then run: !ls /kaggle/input")

def show_samples(paths, n=6, cols=3):
    paths = paths[:: max(len(paths) // n, 1)][:n]
    if not paths:
        print("No images to show.")
        return
    rows = (len(paths) + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 3 * rows))
    for i, p in enumerate(paths):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
        plt.title(os.path.basename(p), fontsize=8)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(IMG_PATHS)


## 3. Test reading a video

**Optional for now.** Your Night Images set is photos, not video, so you can skip to section 4 and come back once you add clips. Upload a short test clip as a Kaggle Dataset (or attach one via **Add Input → Upload**) — you'll see it appear under `/kaggle/input/<dataset-name>/`. No `cv2.imshow` here; we grab one frame and show it inline instead.

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Point this at your uploaded clip once you've attached a dataset, e.g.:
# VIDEO_PATH = "/kaggle/input/beamguard-test-clips/test.mp4"
VIDEO_PATH = f"{BASE}/videos/test.mp4"

def show_first_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        print("Could not read a frame — check the path.")
        return
    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

# show_first_frame(VIDEO_PATH)


In [ ]:
def extract_frames(video_path, out_dir, every_n=10):
    """Pull every Nth frame from a video into out_dir as .jpg — the raw material for your dataset."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    i, saved = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i % every_n == 0:
            cv2.imwrite(os.path.join(out_dir, f"frame_{saved:05d}.jpg"), frame)
            saved += 1
        i += 1
    cap.release()
    print(f"Saved {saved} frames from {video_path}")

# extract_frames(VIDEO_PATH, f"{BASE}/dataset/images/train", every_n=10)


### 3b. Find videos in an attached dataset, and smoke-test the pipeline

After **Add Input** on a video dataset, this lists every video under `/kaggle/input`. If your dataset is image *sequences* instead (like PVDN), `images_to_video` stitches a folder of frames into a clip.

The smoke test uses the pretrained `yolo26n.pt` (cars, motorcycles, buses, trucks only). It has no headlight class yet, so it just proves that reading, detection and tracking work on your clip before you spend GPU time on training.

In [ ]:
import glob

def find_videos(root="/kaggle/input"):
    exts = ("mp4", "avi", "mov", "mkv")
    paths = [p for e in exts for p in glob.glob(f"{root}/**/*.{e}", recursive=True)]
    for p in sorted(paths)[:30]:
        print(p)
    return sorted(paths)

def images_to_video(image_dir, out_path, fps=10):
    files = sorted(glob.glob(f"{image_dir}/*.png") + glob.glob(f"{image_dir}/*.jpg"))
    if not files:
        print("No images found in", image_dir)
        return
    h, w = cv2.imread(files[0]).shape[:2]
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for f in files:
        img = cv2.imread(f)                      # grayscale files load as 3-channel BGR
        writer.write(cv2.resize(img, (w, h)))
    writer.release()
    print("Wrote", out_path, f"({len(files)} frames)")

videos = find_videos()


In [ ]:
from ultralytics import YOLO

def smoke_test(video_path):
    smoke = YOLO("yolo26n.pt")
    for _ in smoke.track(source=video_path, persist=True, stream=True, save=True,
                         classes=[2, 3, 5, 7],
                         project=f"{BASE}/output/processed_videos", name="smoke_test",
                         verbose=False):
        pass
    print("Annotated video saved under", f"{BASE}/output/processed_videos/smoke_test")

# if videos: smoke_test(videos[0])


## 3c. See predictions on your images now (no training needed)

This section predicts on all of your Night Images right away, from the images alone:

- **Vehicles** come from the pretrained `yolo26n.pt`.
- **Headlights and glare** come from a light detector that uses OpenCV rules: very bright spots, how much they glow into the dark around them (lamps bloom, road signs and reflectors do not), lamp pairs, and lane-paint filtering. Each image gets a glare score from 0 to 1 and one verdict.

Be clear about what this is: it is **not a trained model yet**, and the images carry no labels saying which frames are high beam, so nothing here can be scored as right or wrong. I checked the top-ranked frames by eye on your data and they are real oncoming headlights with strong bloom, but the detector will still mislabel some reflectors and street lights. Treat `GLARE_THRESHOLD` as a starting guess and set it by looking at the gallery. The images contain no legible license plates, so no plate can be predicted from this dataset.

The predictions also feed section 4: they become the first-draft labels for training, which you then correct.

In [ ]:
import cv2
import numpy as np

def _halo(gray, x, y, cw, ch):
    # Mean brightness (0-1) of the thin ring just around a light. Lamps glow into the dark
    # around them (high value); signs and reflectors have sharp edges and a dark surround.
    H, W = gray.shape
    pad = max(int(0.5 * max(cw, ch)), 3)
    x0, y0, x1, y1 = max(x - pad, 0), max(y - pad, 0), min(x + cw + pad, W), min(y + ch + pad, H)
    outer = gray[y0:y1, x0:x1].astype(np.float32)
    mask = np.ones(outer.shape, bool)
    mask[y - y0:y - y0 + ch, x - x0:x - x0 + cw] = False
    return float(outer[mask].mean() / 255.0) if mask.any() else 0.0

def detect_lights(frame, core_thr=240, min_area=8, top_ignore=0.35):
    h, w = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    mask = (gray >= core_thr).astype(np.uint8)
    mask[: int(h * top_ignore)] = 0
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    lights = []
    for i in range(1, n):
        x, y, cw, ch, area = [int(v) for v in stats[i]]
        if area < min_area:
            continue
        if max(cw, ch) / max(min(cw, ch), 1) > 3.5 or area / (cw * ch) < 0.45:
            continue                                    # lane paint and other line-like shapes
        px = frame[y:y + ch, x:x + cw].reshape(-1, 3).astype(np.float32)
        b, g, r = px.mean(axis=0)
        halo = _halo(gray, x, y, cw, ch)
        size = min(1.0, area / (0.0006 * h * w))
        bloom = min(1.0, halo / 0.6)
        lights.append({"box": (x, y, x + cw, y + ch), "area": area, "cx": x + cw / 2, "cy": y + ch / 2,
                       "halo": round(halo, 3), "red": bool(r > 1.4 * max(g, b) + 5),
                       "glare": round(0.55 * size + 0.45 * bloom, 3)})
    return lights

def pair_lights(lights):
    used, pairs = set(), []
    cands = []
    for i in range(len(lights)):
        for j in range(i + 1, len(lights)):
            a, b = lights[i], lights[j]
            wmax = max(a["box"][2] - a["box"][0], b["box"][2] - b["box"][0])
            hmax = max(a["box"][3] - a["box"][1], b["box"][3] - b["box"][1])
            dx, dy = abs(a["cx"] - b["cx"]), abs(a["cy"] - b["cy"])
            ratio = max(a["area"], b["area"]) / max(min(a["area"], b["area"]), 1)
            if dy <= 0.8 * hmax and ratio <= 3 and 1.5 * wmax <= dx <= 14 * wmax and a["red"] == b["red"]:
                cands.append((dx, i, j))
    for dx, i, j in sorted(cands):
        if i in used or j in used:
            continue
        used.update((i, j)); pairs.append((i, j))
    return pairs

def predict_lights(frame, big_frac=0.002, bloom_min=0.33, min_head_area=40):
    # Classifies each light. A headlight must show a visible bloom: signs and reflectors
    # have sharp edges and almost no glow, real lamps glow into the dark around them.
    h, w = frame.shape[:2]
    lights = detect_lights(frame)
    pairs = pair_lights(lights)
    paired = {k for p in pairs for k in p}
    for k, L in enumerate(lights):
        glows = L["halo"] >= bloom_min and L["area"] >= min_head_area
        big = L["area"] >= big_frac * h * w
        single_strong = L["halo"] >= 0.45 and L["area"] >= 150
        bw, bh = L["box"][2] - L["box"][0], L["box"][3] - L["box"][1]
        stripe = (max(bw, bh) / max(min(bw, bh), 1) >= 1.6 and L["area"] / (bw * bh) < 0.62) \
            or (bh > 1.3 * bw and L["area"] < big_frac * h * w)      # lane paint: slanted or tall and narrow
        if stripe:
            L["kind"] = "other light"
        elif L["red"] and k in paired:
            L["kind"] = "tail light"
        elif glows and not L["red"] and (k in paired or big or single_strong):
            L["kind"] = "headlight"
        else:
            L["kind"] = "other light"       # signs, reflectors, street lamps, small distant lights
    heads = [L for L in lights if L["kind"] == "headlight"]
    glare = max((L["glare"] for L in heads), default=0.0)
    return lights, glare


In [ ]:
import pandas as pd
from ultralytics import YOLO

GLARE_THRESHOLD = 0.75      # not calibrated: check the gallery below and adjust it by eye
PRED_DIR = f"{BASE}/output/predictions"
os.makedirs(PRED_DIR, exist_ok=True)

coco = YOLO("yolo26n.pt")   # pretrained: finds cars, motorcycles, buses, trucks
VEHICLE_IDS = {2, 3, 5, 7}
COLORS = {"headlight": (0, 255, 0), "tail light": (0, 0, 255), "other light": (255, 160, 0)}

def verdict(n_vehicles, n_heads, glare):
    if n_heads == 0 and n_vehicles == 0:
        return "no vehicle or headlights found"
    if glare >= GLARE_THRESHOLD:
        return "strong glare (possible high beam)"
    if n_heads > 0:
        return "headlights, normal glare"
    return "vehicle found, no lit headlights"

def predict_image(path, save=True):
    frame = cv2.imread(path)
    lights, glare = predict_lights(frame)
    r = coco.predict(frame, conf=0.25, verbose=False)[0]
    vehicles = [b.xyxy[0].tolist() for b in r.boxes if int(b.cls[0]) in VEHICLE_IDS]
    n_heads = sum(l["kind"] == "headlight" for l in lights)
    result = {
        "image": os.path.basename(path),
        "vehicles": len(vehicles),
        "headlights": n_heads,
        "glare_score": round(glare, 3),
        "verdict": verdict(len(vehicles), n_heads, glare),
    }
    if save:
        canvas = frame.copy()
        for x1, y1, x2, y2 in vehicles:
            cv2.rectangle(canvas, (int(x1), int(y1)), (int(x2), int(y2)), (255, 255, 0), 3)
        for l in lights:
            x1, y1, x2, y2 = l["box"]
            cv2.rectangle(canvas, (x1 - 6, y1 - 6), (x2 + 6, y2 + 6), COLORS[l["kind"]], 3)
        cv2.putText(canvas, f"{result['verdict']} | glare {glare:.2f}", (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
        cv2.imwrite(os.path.join(PRED_DIR, result["image"]), canvas)
    return result

predictions = pd.DataFrame([predict_image(p) for p in IMG_PATHS])
predictions = predictions.sort_values("glare_score", ascending=False).reset_index(drop=True)
predictions.to_csv(f"{BASE}/output/predictions.csv", index=False)

print(f"Predicted {len(predictions)} images. Saved predictions.csv and annotated images in {PRED_DIR}")
print(predictions["verdict"].value_counts())
predictions.head(15)


In [ ]:
def show_predictions(df, n=6, cols=2):
    names = df["image"].tolist()[:n]
    if not names:
        print("Nothing to show.")
        return
    rows = (len(names) + cols - 1) // cols
    plt.figure(figsize=(9 * cols, 5.5 * rows))
    for i, name in enumerate(names):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(cv2.cvtColor(cv2.imread(os.path.join(PRED_DIR, name)), cv2.COLOR_BGR2RGB))
        plt.title(name)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_predictions(predictions, n=6)     # the strongest-glare frames
# show_predictions(predictions[predictions["verdict"] == "headlights, normal glare"], n=6)
# show_predictions(predictions[predictions["verdict"] == "no vehicle or headlights found"], n=6)


## 4. Prepare the Night Images set for training

What is in this dataset, and what it means for the project:

- **400 JPGs but only about 106 distinct frames.** Files like `frame_0001.jpg`, `frame_0001(1).jpg` ... `frame_0001(4).jpg` are near-duplicates of the same scene. If you split randomly, copies of the same scene land in both train and validation and your scores look better than they really are. The split below keeps each frame number together.
- **No labels.** YOLO needs boxes for `0: vehicle` and `1: headlight`.
- **Mostly empty dark road.** Many frames have no vehicle at all. That is useful as negative examples, but the number of frames with real headlights is small.

So this set is enough to build and test the pipeline, and to start labeling. For a detector you can trust, add more footage later (your own clips, PVDN frames).

The steps: (4a) split without leakage, (4b) auto-draw first-draft labels (vehicles from the pretrained model, headlights from the light detector in 3c), (4c) correct them in an annotation tool, (4d) point `dataset.yaml` at the result.

In [ ]:
import re, random, shutil
from collections import defaultdict

def frame_group(path):
    m = re.search(r"frame_([0-9]+)", os.path.basename(path))
    return m.group(1) if m else os.path.basename(path)

def make_splits(img_paths, ratios=(0.70, 0.15, 0.15), seed=42):
    groups = defaultdict(list)
    for p in img_paths:
        groups[frame_group(p)].append(p)

    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    n_train = int(len(keys) * ratios[0])
    n_val = int(len(keys) * ratios[1])
    split_keys = {
        "train": keys[:n_train],
        "val": keys[n_train:n_train + n_val],
        "test": keys[n_train + n_val:],
    }

    counts = {}
    for split, ks in split_keys.items():
        counts[split] = 0
        for k in ks:
            for p in groups[k]:
                clean = re.sub(r"[()]", "_", os.path.basename(p))
                shutil.copy(p, os.path.join(BASE, "dataset", "images", split, clean))
                counts[split] += 1
    print("Images per split:", counts)

make_splits(IMG_PATHS)


In [ ]:
def yolo_line(cls_id, x1, y1, x2, y2, w, h):
    x1, x2 = max(x1, 0), min(x2, w)
    y1, y2 = max(y1, 0), min(y2, h)
    return f"{cls_id} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}"

def prelabel_all(conf=0.25, pad=4):
    # Auto-draws boxes so you only have to correct them (needs the cells in 3c to have run):
    #   class 0 vehicle   <- pretrained detector (COCO car, motorcycle, bus, truck)
    #   class 1 headlight <- the light detector from 3c (only lights it classed as headlights)
    det = YOLO("yolo26n.pt")
    vehicle_ids = {2, 3, 5, 7}
    with_labels = []

    for split in ("train", "val", "test"):
        img_dir = f"{BASE}/dataset/images/{split}"
        lbl_dir = f"{BASE}/dataset/labels/{split}"
        for p in sorted(glob.glob(img_dir + "/*.jpg")):
            frame = cv2.imread(p)
            h, w = frame.shape[:2]
            lines = []
            r = det.predict(frame, conf=conf, verbose=False)[0]
            for b in r.boxes:
                if int(b.cls[0]) in vehicle_ids:
                    x1, y1, x2, y2 = b.xyxy[0].tolist()
                    lines.append(yolo_line(0, x1, y1, x2, y2, w, h))
            lights, _ = predict_lights(frame)
            for l in lights:
                if l["kind"] == "headlight":
                    x1, y1, x2, y2 = l["box"]
                    lines.append(yolo_line(1, x1 - pad, y1 - pad, x2 + pad, y2 + pad, w, h))
            name = os.path.splitext(os.path.basename(p))[0] + ".txt"
            with open(os.path.join(lbl_dir, name), "w") as f:
                f.write("\n".join(lines))      # an empty file = a valid "no objects" example
            if lines:
                with_labels.append(p)

    print(f"{len(with_labels)} of {len(glob.glob(BASE + '/dataset/images/*/*.jpg'))} images got at least one auto-drawn box")
    return with_labels

with_labels = prelabel_all()
show_samples(with_labels, n=6)


### 4c. Correct the first-draft labels

1. Run the next cell to zip `dataset/` (images + the auto-drawn vehicle labels).
2. Download `to_review.zip` from the notebook's **Output** tab (after you Save Version) and import it into a free annotation tool that reads YOLO format, such as Roboflow or CVAT.
3. Delete wrong boxes (reflectors, signs and street lamps marked as headlights), add missed vehicles and headlights, and keep class `0: vehicle` and `1: headlight`. Start with the images in `with_labels`.
4. Export in **YOLO format**, upload the export as a new private Kaggle Dataset (for example `night-images-labeled`), and attach it with **Add Input**.
5. Set `DATASET_ROOT` in the next-next cell to that dataset's folder.

You can train right away on the first-draft labels to get a working model. Expect it to mostly repeat the light detector's guesses, including its mistakes, so it improves only as you correct the labels.

In [ ]:
shutil.make_archive("/kaggle/working/to_review", "zip", f"{BASE}/dataset")
print("Wrote /kaggle/working/to_review.zip", round(os.path.getsize("/kaggle/working/to_review.zip") / 1e6, 1), "MB")


In [ ]:
# Until you have corrected labels, this points at the auto-drawn first-draft ones.
# After the review, change it to your labeled dataset, e.g. "/kaggle/input/night-images-labeled".
# If the export uses folders like train/images and valid/images, edit train/val/test below to match.
DATASET_ROOT = f"{BASE}/dataset"

dataset_yaml = f"""
path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: vehicle
  1: headlight
"""

yaml_path = f"{BASE}/dataset.yaml"
with open(yaml_path, "w") as f:
    f.write(dataset_yaml)

print(dataset_yaml)


## 5. Train the vehicle/headlight detector

`yolo26n.pt` (YOLO26 Nano, Ultralytics' current smallest/fastest model, released Jan 2026) is a good starting checkpoint for transfer learning. `yolo11n.pt` is an equally valid, slightly older alternative if you hit compatibility issues.

Mind the GPU clock: a session caps out around 9–12 hours and your weekly quota is ~30 hours, so for the first run keep `epochs` modest (e.g. 50) and check `time=` if you want Ultralytics to auto-stop after a fixed duration instead of epochs.

In [ ]:
import torch
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

RUN_TRAINING = True   # set to False before "Save & Run All" if you don't want to spend GPU hours

if RUN_TRAINING:
    results = model.train(
        data=yaml_path,
        epochs=50,
        imgsz=640,
        device=0 if torch.cuda.is_available() else "cpu",
        project=f"{BASE}/models",
        name="vehicle_headlight_detector",
    )


**Persist your model before the session ends** — `/kaggle/working` only survives if you click **Save Version (Commit)**. The trained weights land at `{BASE}/models/vehicle_headlight_detector/weights/best.pt`; that path becomes an Output you can download or attach as a new Kaggle Dataset to reuse in later notebooks without retraining.

## 6. Headlight brightness / glare analysis

Once YOLO gives you a headlight bounding box, crop it and measure it with OpenCV.

In [ ]:
import numpy as np

def analyze_headlight_roi(frame, box):
    """box = (x1, y1, x2, y2) in pixel coords. Returns brightness/glare features."""
    x1, y1, x2, y2 = [int(v) for v in box]
    roi = frame[y1:y2, x1:x2]
    if roi.size == 0:
        return None

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    mean_brightness = float(gray.mean())
    max_brightness = float(gray.max())
    bright_pixels = int((gray > 200).sum())
    total_pixels = gray.size
    bright_ratio = bright_pixels / total_pixels

    return {
        "mean_brightness": mean_brightness,
        "max_brightness": max_brightness,
        "bright_ratio": bright_ratio,
    }


Don't hard-code a `brightness > 200 = violation` threshold — camera exposure shifts pixel values too much. Log these features across known low-beam and high-beam clips first, and only pick thresholds once you can see the actual separation in your own data.

## 7. High-beam score

Combine multiple features into one score once you've measured what actually separates your classes.

In [ ]:
def high_beam_score(features, weights=None):
    if features is None:
        return 0.0
    weights = weights or {"intensity": 0.40, "glare": 0.25, "beam": 0.20, "persistence": 0.15}

    intensity_score = min(features["mean_brightness"] / 255, 1.0)
    glare_score = min(features["bright_ratio"] * 3, 1.0)     # scale factor: tune against your data
    beam_score = min(features["max_brightness"] / 255, 1.0)
    # persistence_score is filled in per-track in the tracking step below
    persistence_score = features.get("persistence_score", 0.0)

    score = (
        weights["intensity"] * intensity_score
        + weights["glare"] * glare_score
        + weights["beam"] * beam_score
        + weights["persistence"] * persistence_score
    )
    return round(score, 3)


## 7b. Test on still images

You can check single images (or a whole folder) with your model. To get images into Kaggle: right panel → **Input → Upload → New Dataset**, drop your images in, and they appear at `/kaggle/input/<your-dataset-slug>/`.

Before you've trained anything, you can still sanity-check with the pretrained `yolo26n.pt` (it detects cars/trucks/buses, just not headlights). After training, point `WEIGHTS` at your `best.pt`.

In [ ]:
import glob
from ultralytics import YOLO

# Use your trained weights once training is done; falls back to the pretrained model until then
found = glob.glob(f"{BASE}/models/**/weights/best.pt", recursive=True)
WEIGHTS = found[0] if found else "yolo26n.pt"
check_model = YOLO(WEIGHTS)
print("Using weights:", WEIGHTS)

def check_image(image_path, conf=0.25):
    frame = cv2.imread(image_path)
    if frame is None:
        print("Could not read image:", image_path)
        return
    r = check_model.predict(frame, conf=conf, verbose=False)[0]
    annotated = frame.copy()

    for box in r.boxes:
        cls_id = int(box.cls[0])
        name = r.names[cls_id]
        xyxy = [int(v) for v in box.xyxy[0].tolist()]
        label = f"{name} {float(box.conf[0]):.2f}"

        if name == "headlight":
            features = analyze_headlight_roi(frame, xyxy)
            if features:
                label += f" | score {high_beam_score(features):.2f}"

        cv2.rectangle(annotated, xyxy[:2], xyxy[2:], (0, 255, 0), 2)
        cv2.putText(annotated, label, (xyxy[0], max(xyxy[1] - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(os.path.basename(image_path))
    plt.show()

def check_folder(folder, limit=10):
    paths = sorted(glob.glob(os.path.join(folder, "*.jpg")) + glob.glob(os.path.join(folder, "*.png")))[:limit]
    for p in paths:
        check_image(p)

# check_image(IMG_PATHS[0])
# check_folder(DATA_DIR, limit=10)


A single image has no history, so the persistence term in the score stays at 0 (the score tops out at 0.85). That's fine for eyeballing whether detections and brightness values look sensible; the persistence check only applies in the video/tracking pipeline below.

## 8. Vehicle tracking + persistence

YOLO's built-in tracker (ByteTrack under the hood) gives each vehicle a stable ID across frames, so a bright reading only counts once it holds up over several consecutive frames — not a single flicker.

In [ ]:
from collections import defaultdict, deque

track_history = defaultdict(lambda: deque(maxlen=10))  # last 10 scores per track id
PERSISTENCE_FRAMES = 5
SCORE_THRESHOLD = 0.75

def update_track(track_id, score):
    track_history[track_id].append(score)
    recent = list(track_history[track_id])[-PERSISTENCE_FRAMES:]
    is_candidate = (
        len(recent) == PERSISTENCE_FRAMES
        and all(s > SCORE_THRESHOLD for s in recent)
    )
    return is_candidate

# Example usage over a video with tracking enabled:
# results = model.track(source=VIDEO_PATH, persist=True, stream=True)
# for r in results:
#     for box in r.boxes:
#         track_id = int(box.id) if box.id is not None else None
#         ...


## 9. License plate + OCR (separate module)

Keep this decoupled from the detector — don't try to make one model do vehicle + headlight + plate + OCR from the start. `EasyOCR` tends to install more cleanly on Kaggle than PaddleOCR; swap in PaddleOCR later if you want its accuracy and don't mind the extra setup.

In [ ]:
import torch
import easyocr

reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

def read_plate(plate_roi):
    results = reader.readtext(plate_roi)
    if not results:
        return None, 0.0
    text, confidence = max(((t, c) for _, t, c in results), key=lambda x: x[1])
    return text, float(confidence)


## 10. Evidence generation + SQLite log

In [ ]:
import sqlite3
from datetime import datetime

DB_PATH = f"{BASE}/output/violations.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS violations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            vehicle_track_id INTEGER,
            plate_number TEXT,
            timestamp TEXT,
            camera_id TEXT,
            mean_brightness REAL,
            bright_ratio REAL,
            violation_score REAL,
            image_path TEXT,
            review_status TEXT DEFAULT 'pending'
        )
    """)
    conn.commit()
    conn.close()

def log_violation(track_id, plate_number, features, score, image_path, camera_id="CAM01"):
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        """INSERT INTO violations
           (vehicle_track_id, plate_number, timestamp, camera_id, mean_brightness, bright_ratio, violation_score, image_path)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
        (track_id, plate_number, datetime.now().isoformat(), camera_id,
         features["mean_brightness"], features["bright_ratio"], score, image_path),
    )
    conn.commit()
    conn.close()

init_db()


In [ ]:
def save_evidence_image(frame, vehicle_box, headlight_box, plate_box, plate_text, score, out_dir):
    """Draws boxes + labels and writes the evidence frame to disk. No GUI needed — imwrite works headless."""
    annotated = frame.copy()
    for box, color, label in [
        (vehicle_box, (0, 255, 0), "vehicle"),
        (headlight_box, (0, 200, 255), "headlight"),
        (plate_box, (255, 0, 0), plate_text or "plate"),
    ]:
        if box is None:
            continue
        x1, y1, x2, y2 = [int(v) for v in box]
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        cv2.putText(annotated, label, (x1, max(y1 - 8, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.putText(annotated, f"HIGH-BEAM CANDIDATE: {score:.0%}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = os.path.join(out_dir, f"violation_{ts}.jpg")
    cv2.imwrite(out_path, annotated)
    return out_path


## 11. Full pipeline skeleton

Ties the pieces together over a video. This is a starting skeleton, not a finished system — you'll extend it as tracking, plate detection, and scoring get tuned on your own data.

In [ ]:
def run_pipeline(video_path, evidence_dir=f"{BASE}/output/evidence"):
    os.makedirs(evidence_dir, exist_ok=True)
    results = model.track(source=video_path, persist=True, stream=True, verbose=False)

    for r in results:
        frame = r.orig_img
        if r.boxes is None:
            continue
        for box in r.boxes:
            cls_id = int(box.cls[0])
            track_id = int(box.id[0]) if box.id is not None else None
            xyxy = box.xyxy[0].tolist()

            if cls_id != 1 or track_id is None:   # class 1 = headlight, from dataset.yaml
                continue

            features = analyze_headlight_roi(frame, xyxy)
            if features is None:
                continue

            score = high_beam_score(features)
            is_candidate = update_track(track_id, score)

            if is_candidate:
                image_path = save_evidence_image(frame, None, xyxy, None, None, score, evidence_dir)
                log_violation(track_id, None, features, score, image_path)
                print(f"Logged candidate — track {track_id}, score {score:.2f} -> {image_path}")

# run_pipeline(VIDEO_PATH)


## 12. Dashboard note

Streamlit needs a live server with an open port, which a Kaggle notebook doesn't give you — running `streamlit run` here won't produce anything you can reach. Two practical options:

1. **Inline report (works right here)** — a quick pandas view of `violations.db`, good enough to demo inside the notebook.
2. **Real dashboard, run locally** — after you commit this notebook, download `output/violations.db` and `output/evidence/` from the notebook's *Output* tab, then run the actual Streamlit app (`streamlit run dashboard/app.py`) on your own machine or cPanel, pointed at the downloaded files.

In [ ]:
import pandas as pd

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM violations ORDER BY timestamp DESC", conn)
conn.close()

print(f"Total logged candidates: {len(df)}")
df.head(20)


To download everything the dashboard needs in one go, bundle the results into a single zip, then download `results.zip` from the Output panel. Unzip it next to `dashboard/app.py` on your computer, so you end up with `output/violations.db` and `output/evidence/`.

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/results", "zip", f"{BASE}/output")
print("Wrote /kaggle/working/results.zip", round(os.path.getsize("/kaggle/working/results.zip") / 1e6, 1), "MB")


## Suggested week-by-week plan (Kaggle-adjusted)

| Week | Focus | Kaggle-specific notes |
|---|---|---|
| 1 | Python/OpenCV refresher, video → frame reading | Use the matplotlib-based `show_first_frame`, not `cv2.imshow` |
| 2 | Vehicle detection with a pretrained model | Run with GPU accelerator only while actually testing inference |
| 3 | Collect nighttime footage | Done outside Kaggle; upload clips as a private Dataset when ready |
| 4 | Annotate vehicles + headlights | Annotate elsewhere (Roboflow/CVAT), export YOLO format, upload as a Dataset |
| 5 | Train first custom YOLO26 model | Watch the 9–12h session cap and ~30h/week quota; commit to save `best.pt` |
| 6 | Brightness/glare analysis | Pure CPU work — turn the accelerator off to save quota |
| 7 | High-beam scoring | Tune weights/thresholds against your own logged brightness values |
| 8 | Vehicle tracking | Use `model.track(persist=True)`; test on a committed run so it survives session end |
| 9 | License plate + OCR | EasyOCR install is lighter-weight on Kaggle than PaddleOCR |
| 10 | Evidence generation + SQLite | Everything writes under `/kaggle/working/BeamGuardAI/output` |
| 11 | Dashboard | Build the inline pandas report here; run the real Streamlit app locally |
| 12 | Evaluation (precision/recall/mAP/F1) + writeup | `model.val()` after training gives you mAP directly from Ultralytics |

Each week's cells above are meant to be run, tweaked, and committed independently — you don't need to keep one giant session alive for the whole project.